# 一些Trick

## 划分训练集和测试集

In [ ]:
# 划分训练集和测试集
# 方法一：对于使用sklearn的机器学习任务
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# 方法二：对于使用torch的深度学习任务
from torch.utils.data import random_split
dataset = MyDataset(dataset_config)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

## 快速调参法

In [ ]:
# 回归任务pipeline
# 使用网格搜索半自动调参
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

# 构建Pipeline，包含数据标准化和回归模型
pipeline = make_pipeline(StandardScaler(), Ridge())

# 定义需要搜索的参数网格
# 注意：参数的键名格式为 '模型名称__参数名称'，这里模型名称是 'ridge'，参数名称是 'alpha'
# 比如 如果是 RandomForestRegressor 模型，参数名称是 'n_estimators'，则键名应该是 'randomforestregressor__n_estimators'
# 如果是lasso模型，参数名称是 'alpha'，则键名应该是 'lasso__alpha'
param_grid = {
    'ridge__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]  # 注意参数的键名格式
}

# 创建GridSearchCV对象，进行5折交叉验证，使用负均方误差作为评分标准
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='neg_mean_squared_error')
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation score (neg MSE):", grid_search.best_score_)

# 使用最佳参数训练最终模型
best_pipeline = grid_search.best_estimator_
best_pipeline.fit(X_train, y_train)
y_pred = best_pipeline.predict(X_test)


In [ ]:
# 分类任务pipeline
# 快速搭建一个k-折交叉验证进行手动调参
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
pipeline = make_pipeline(
    TfidfVectorizer(max_features=5000, ngram_range=(1,2), max_df=0.8, min_df=3),
    LinearSVC(C=0.0000001, class_weight='balanced')
)
scores = cross_val_score(pipeline, train_text, label, cv=5)
print("CV accuracy: {:.4f}".format(scores.mean()))

# 可以根据cv结果调整模型的参数，一般来说在蓝桥杯中
# 在训练集上应该达到和题目要求差不多的分数会比较好，太高可能就过拟合了

# 基础回归和分类模型


## 基础回归模型

In [ ]:
from sklearn import linear_model
# 常见的回归任务模型：linear regression, ridge regression, lasso regression
# 基础的调用框架如下：
model = linear_model.LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# ridge regression 是在线性回归的基础上增加了L2正则化项，可以防止过拟合，适用于特征较多且存在多重共线性的情况
model = linear_model.Ridge(alpha=1.0)
# 其中alpha是正则化强度的参数，值越大表示正则化越强，默认为1.0


# lasso regression 是在线性回归的基础上增加了L1正则化项，可以进行特征选择，适用于特征较多且希望得到稀疏模型的情况
model = linear_model.Lasso(alpha=1.0)

In [ ]:
# 对于回归任务的评估指标，常用的有均方误差（MSE）、平均绝对误差（MAE）和R²分数等
# 其中，MSE越小表示模型的预测误差越小，MAE越小表示模型的平均预测误差越小，R²分数越接近1表示模型的拟合程度越好
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

## 基础分类模型

In [ ]:
from sklearn.svm import LinearSVC, SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
# 常见的分类任务模型：SVM, Logistic Regression, Random Forest, XGBoosting
# 小型数据可以特征提取器套 SVM
# LinearSVC：适用于较大规模，高维稀疏特征的数据（比如文本分类）
# SVC：适用于较小规模，非线性可分的数据，支持多种核函数（比如RBF核）
# Logistic Regression：适用于线性可分的数据，输出概率，适合二分类和多分类任务
# Random Forest：适用于各种类型的数据，能够处理非线性关系，具有较好的泛化能力，适合特征重要性分析
# XGBoosting：适用于各种类型的数据，能够处理非线性关系，具有较好的泛化能力，适合大规模数据和特征重要性分析


# 基础的调用框架如下：
model = LinearSVC(C=0.0000001, class_weight='balanced') 
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
# C是正则化参数，值越小表示正则化越强，越不容易过拟合，默认为1.0
# 对于类别不平衡的数据可设置class_weight='balanced'

# SVC的一些常用参数：
# kernel：核函数类型，常用的有'linear'（线性核）、'poly'（多项式核）、'rbf'（径向基函数核）和'sigmoid'（sigmoid核），默认为'rbf'
# degree：多项式核函数的阶数，仅在kernel='poly'时有效
model = SVC(kernel='rbf', degree=3, C=1.0, class_weight='balanced')

model = LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000)

model = RandomForestClassifier(n_estimators=100, max_depth=2, random_state=42)

# TODO: 记录调参经验

In [ ]:
# 对于分类任务的评估指标，常用的有准确率（accuracy）、精确率（precision）、召回率（recall）和F1分数等
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')